[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/signal38/signal38.github.io/blob/main/notebooks/01_baseline.ipynb)

# Notebook 01 — Baseline Models

Trains two baselines:
1. **Naive rule** — maps `avg_goldstein` to escalation level via fixed thresholds
2. **XGBoost** — trained on 13-element GDELT feature vectors

Runs on CPU only. Runtime ~5 min.

**Outputs (published to `main`):**
- `models/xgb/model.json` — trained XGBoost model
- `data/outputs/baseline_results.json` — MAE and accuracy for both models on val and test sets

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/signal38.github.io')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/signal38/signal38.github.io.git', str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import ensure_notebook_requirements
ensure_notebook_requirements('01_baseline', requirements_path=str(REPO / 'requirements.txt'))
# Runtime may restart after this cell.
# After restart, run all cells again — the sentinel skips reinstall.


In [ ]:
import subprocess, sys, os, json, re
from pathlib import Path

REPO = Path('/content/signal38.github.io')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/signal38/signal38.github.io.git', str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import prepare_notebook
REPO, PATHS = prepare_notebook(REPO, pull_latest=True)
print(f'Repo ready at {REPO}')


In [ ]:
import json, re
import numpy as np
from scripts.features import cluster_to_features, FEATURE_NAMES
from scripts.metrics import goldstein_to_escalation, escalation_mae, escalation_accuracy


def load_split(split_path, clusters_path):
    clusters = {}
    for line in Path(clusters_path).read_text().splitlines():
        if line.strip():
            c = json.loads(line)
            clusters[c['week_start']] = c

    X_rows, y_rows, weeks = [], [], []
    for line in Path(split_path).read_text().splitlines():
        if not line.strip():
            continue
        ex = json.loads(line)
        user_msg = ex['messages'][1]['content']
        m = re.search(r'Week: (\d{4}-\d{2}-\d{2})', user_msg)
        if not m:
            continue
        week = m.group(1)
        try:
            asst = json.loads(ex['messages'][2]['content'])
            level = int(asst['escalation_level'])
        except (json.JSONDecodeError, KeyError, ValueError):
            continue
        if week not in clusters:
            continue
        X_rows.append(cluster_to_features(clusters[week]))
        y_rows.append(level)
        weeks.append(week)

    return np.array(X_rows, dtype=np.float32), np.array(y_rows, dtype=np.int32), weeks


X_train, y_train, weeks_train = load_split(PATHS['training_dir'] / 'train.jsonl', PATHS['clusters'])
X_val,   y_val,   weeks_val   = load_split(PATHS['training_dir'] / 'val.jsonl',   PATHS['clusters'])
X_test,  y_test,  weeks_test  = load_split(PATHS['training_dir'] / 'test.jsonl',  PATHS['clusters'])

print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')
print(f'Label distribution (train): {dict(zip(*np.unique(y_train, return_counts=True)))}')

In [ ]:
# avg_goldstein is feature index 0
naive_val_preds  = [goldstein_to_escalation(x[0]) for x in X_val]
naive_test_preds = [goldstein_to_escalation(x[0]) for x in X_test]

naive_val_mae   = escalation_mae(naive_val_preds,  y_val.tolist())
naive_val_acc   = escalation_accuracy(naive_val_preds,  y_val.tolist())
naive_test_mae  = escalation_mae(naive_test_preds, y_test.tolist())
naive_test_acc  = escalation_accuracy(naive_test_preds, y_test.tolist())

print('Naive Goldstein baseline')
print(f'  Val  — MAE: {naive_val_mae:.3f}, Accuracy: {naive_val_acc:.3f}')
print(f'  Test — MAE: {naive_test_mae:.3f}, Accuracy: {naive_test_acc:.3f}')

In [ ]:
import xgboost as xgb

dtrain = xgb.DMatrix(X_train, label=y_train - 1)  # labels 0-indexed for XGB
dval   = xgb.DMatrix(X_val,   label=y_val   - 1)
dtest  = xgb.DMatrix(X_test,  label=y_test  - 1)

params = {
    'objective':        'multi:softmax',
    'num_class':        5,
    'max_depth':        4,
    'eta':              0.1,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'eval_metric':      'mlogloss',
    'seed':             42,
}

evals = [(dtrain, 'train'), (dval, 'val')]
model = xgb.train(
    params, dtrain,
    num_boost_round=200,
    evals=evals,
    early_stopping_rounds=20,
    verbose_eval=20,
)
print(f'Best round: {model.best_iteration}, best val mlogloss: {model.best_score:.4f}')

In [ ]:
# Predictions are 0-indexed; add 1 to restore 1-5 scale
xgb_val_preds  = (model.predict(dval).astype(int)  + 1).tolist()
xgb_test_preds = (model.predict(dtest).astype(int) + 1).tolist()

xgb_val_mae   = escalation_mae(xgb_val_preds,  y_val.tolist())
xgb_val_acc   = escalation_accuracy(xgb_val_preds,  y_val.tolist())
xgb_test_mae  = escalation_mae(xgb_test_preds, y_test.tolist())
xgb_test_acc  = escalation_accuracy(xgb_test_preds, y_test.tolist())

print(f'{"Model":<20} {"Val MAE":>10} {"Val Acc":>10} {"Test MAE":>10} {"Test Acc":>10}')
print('-' * 62)
print(f'{"Naive baseline":<20} {naive_val_mae:>10.3f} {naive_val_acc:>10.3f} {naive_test_mae:>10.3f} {naive_test_acc:>10.3f}')
print(f'{"XGBoost":<20} {xgb_val_mae:>10.3f} {xgb_val_acc:>10.3f} {xgb_test_mae:>10.3f} {xgb_test_acc:>10.3f}')

In [ ]:
import pandas as pd
scores = model.get_fscore()
importance = pd.DataFrame({'feature': list(scores.keys()), 'importance': list(scores.values())})
importance = importance.sort_values('importance', ascending=False)
print(importance.to_string(index=False))

In [ ]:
xgb_dir = PATHS['xgb_dir']
xgb_dir.mkdir(parents=True, exist_ok=True)
model.save_model(str(xgb_dir / 'model.json'))
print(f'Saved to {xgb_dir / "model.json"}')

In [ ]:
from scripts.metrics import save_results

results = {
    'naive_baseline': {
        'val_mae':      naive_val_mae,
        'val_accuracy': naive_val_acc,
        'test_mae':     naive_test_mae,
        'test_accuracy':naive_test_acc,
    },
    'xgboost': {
        'val_mae':      xgb_val_mae,
        'val_accuracy': xgb_val_acc,
        'test_mae':     xgb_test_mae,
        'test_accuracy':xgb_test_acc,
        'best_round':   int(model.best_iteration),
    },
}
save_results(results, PATHS['outputs_dir'] / 'baseline_results.json')

In [ ]:
from scripts.colab_utils import publish_artifacts

publish_artifacts(
    paths=[
        'models/xgb/model.json',
        'data/outputs/baseline_results.json',
    ],
    message='Add XGBoost baseline model and evaluation results',
    repo_dir=REPO,
)